In [ ]:
#!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import json, numpy as np, pandas as pd

REPO_ID = "l3mon3/embedded_vector"

corpus_path = hf_hub_download(repo_id=REPO_ID, filename="corpus.json", repo_type="dataset")
emb_path = hf_hub_download(repo_id=REPO_ID, filename="embeddings.npy", repo_type="dataset")
qemb_path = hf_hub_download(repo_id=REPO_ID, filename="query_embeddings.npy", repo_type="dataset")
test_path = hf_hub_download(repo_id=REPO_ID, filename="test-00000-of-00001.parquet", repo_type="dataset")

corpus = json.load(open(corpus_path))
embeddings = np.load(emb_path)
query_embeddings = np.load(qemb_path)
test_df = pd.read_parquet(test_path)

print(len(corpus), embeddings.shape, query_embeddings.shape, test_df.shape)

In [ ]:
import pandas as pd
import os
from pathlib import Path
import json
print(os.getcwd())

# Indexing

In [ ]:
doc_ids = []
documents = []
metadatas = []

for item in corpus:
  enriched_text = f'Văn bản: {item['law_name']}\nNội dung: {item['law_content']}'
  doc_ids.append(item['doc_id'])
  documents.append(enriched_text)
  metadatas.append({'law_name': item['law_name'], 'law_id': item['law_id']})

In [ ]:
%pip install -q chromadb sentence-transformers rank-bm25

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import chromadb

## Model Encoder

In [ ]:
model = SentenceTransformer('BAAI/bge-m3')
# embeddings = model.encode(
#     documents,
#     batch_size=16,
#     show_progress_bar=True,
#     normalize_embeddings=True,
#     #device_map='auto'
# ).tolist()


In [ ]:
db_path = f"{os.getcwd()}/chroma_db"
cilent = chromadb.PersistentClient(path=db_path)

In [ ]:
collection = cilent.get_or_create_collection(
    name='legal_corpus',
    metadata={'hnsw:space':'cosine'}
)
collection.add(
    ids=doc_ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

In [ ]:
import re
def tokenized(text: str):
    text=text.lower()
    text=re.sub(r'[^\w\s]'," ", text)
    return text.split()
tokenized_corpus = [tokenized(doc) for doc in documents]

## Sparse Indexing

In [ ]:
import pickle
bm25 = BM25Okapi(tokenized_corpus)
bm25_data = {
    'bm25': bm25,
    'doc_ids': doc_ids,
    'documents': documents,
    'metadatas': metadatas
}
bm25_path = f"{os.getcwd()}/bm25"
with open(bm25_path,'wb') as f:
    pickle.dump(bm25_data, f)


In [ ]:
corpus_lookup = {(item['law_name'].strip().lower(), item['law_id'].strip().lower()):item['doc_id']
                for item in corpus}


In [ ]:
test_df["target_doc_id"] = [
    corpus_lookup.get((str(name).strip().lower(), str(lid).strip().lower()))
    for name, lid in zip(test_df["law_name"], test_df["law_id"])
]

In [ ]:
def clean_text(text:str) -> str:
    return re.sub(r'\s+', ' ', text).strip()
def extract_components(messages_arr):
    user_query = ""
    ground_truth_answer = ""
    for msg in messages_arr:
        role = msg.get('role')
        if role == 'user':
            user_query = clean_text(msg.get('content', " "))
        elif role == 'assistant':
            ground_truth_answer = clean_text(msg.get('content', " "))
    return user_query, ground_truth_answer

In [ ]:
extracted = [extract_components(arr) for arr in test_df['messages']]


In [ ]:
test_df['query_to_embed'] = [item[0] for item in extracted]
test_df['ground_truth_ans'] = [item[1] for item in extracted]

In [ ]:
queries = test_df['query_to_embed'].tolist()
# query_embeddings =  model.encode(
#     queries,
#     batch_size=16,
#     show_progress_bar=True,
#     normalize_embeddings=True,
#     #device_map='auto'
# ).tolist()
# np.save('query_embeddings.npy', np.array(query_embeddings,dtype=np.float32))

## No Ranking Search

In [ ]:
def search_dense_with_embedding(query_emb: list[float], top_k: int = 20):
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=['documents', 'metadatas','distances']
    )
    dense_hits = []
    if results and results['ids']:
        for i, doc_id in enumerate(results['ids'][0]):
            dense_hits.append({
                'doc_id': doc_id,
                'document': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'score': 1.0 - results['distances'][0][i]
            })
    return dense_hits

def search_sparse(query: str, top_k: int = 20):
    tokenized_query = tokenized(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    sparse_hits=[]
    for idx in top_indices:
        if scores[idx] > 0:
            sparse_hits.append({
                'doc_id': doc_ids[idx],
                'document': documents[idx],
                'metadata': metadatas[idx],
                'score': float(scores[idx])
            })
    return sparse_hits


# Generation

In [ ]:
from sentence_transformers import CrossEncoder
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda:1")

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
#Quantization config
# nf4_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16,   # đổi từ bfloat16 sang float16 — T4 hỗ trợ native
# )


In [ ]:
%pip install -U -q bitsandbytes

In [ ]:
llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME,
                                             #quantization_config=nf4_config,
                                             low_cpu_mem_usage=True, device_map={'': 0})

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

In [ ]:
%pip install -q rouge-score nltk transformers accelerate

In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 512) -> str:
    """Hàm inference sinh phản hồi từ LLM"""
    messages = [
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    inputs = llm_tokenizer(formatted_prompt, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
        )

    # Take the answer only
    generated_text = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

def build_rag_prompt(query: str, retrieved_docs: list[dict]) -> str:
    """Ghép ngữ cảnh và chỉ định format đầu ra cho LLM"""
    context_blocks = []
    for i, doc in enumerate(retrieved_docs, start=1):
        context_blocks.append(f"--- [Tài liệu {i}] ---\n{doc['document']}")

    context_str = "\n\n".join(context_blocks)

    prompt = f"""Bạn là một chuyên gia tư vấn pháp luật. Dựa trên các quy định của pháp luật Việt Nam được cung cấp dưới đây, hãy giải đáp câu hỏi của người dùng và BẮT BUỘC trình bày theo đúng định dạng chuẩn 3 phần sau:

**Căn cứ pháp lý:** [Tên điều luật] - [Tên văn bản luật]

**Nội dung quy định:**
[Trích dẫn chính xác nội dung điều luật áp dụng]

**Phân tích & Hướng dẫn:**
[Phân tích, áp dụng quy định trên vào tình huống của người dùng để trả lời câu hỏi]

---
CĂN CỨ PHÁP LUẬT THAM KHẢO:
{context_str}

CÂU HỎI:
{query}

TRẢ LỜI:"""
    return prompt



In [ ]:
import re
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def check_citation_accuracy(pred_text: str, gt_law_name: str, gt_law_id: str) -> bool:
    """
    Kiểm tra xem câu trả lời có chứa đúng tên luật và điều luật chuẩn hay không
    """
    pred_lower = pred_text.lower()

    # Normalize search
    law_name_clean = gt_law_name.strip().lower()
    law_id_clean = gt_law_id.strip().lower()

    # Condition: must have ID and artical name
    has_law_id = law_id_clean in pred_lower
    has_law_name = law_name_clean in pred_lower

    return bool(has_law_id and has_law_name)


def check_format_adherence(pred_text: str) -> float:
    """
    Evaluate format need to adhere to following points:
    1. **Căn cứ pháp lý:**
    2. **Nội dung quy định:**
    3. **Phân tích & Hướng dẫn:**
    return ratio format followed: 0.0, 0.33, 0.67, or 1.0
    """
    patterns = [
        r"\*\*căn cứ pháp lý:?\*\*",
        r"\*\*nội dung quy định:?\*\*",
        r"\*\*phân tích\s*(&|và)\s*hướng dẫn:?\*\*"
    ]

    matched_count = 0
    for pattern in patterns:
        if re.search(pattern, pred_text, flags=re.IGNORECASE):
            matched_count += 1

    return matched_count / len(patterns)

In [ ]:
def hybrid_search_ablation(
    query_text: str,
    query_emb: list = None,
    top_k: int = 5,
    fetch_k: int = 25,
    k_rrf: int = 60,
    dense_weight: float = 1,
    use_dense: bool = True,
    use_sparse: bool = True,
    use_rerank: bool = True,
):
    """
    Retrieval for ABLATION STUDY.
    """
    if not use_dense and not use_sparse:
        return []

    if use_dense and query_emb is None:
        query_emb = model.encode([query_text], normalize_embeddings=True)[0].tolist()
    #print('dense_weight:', dense_weight)
    dense_results = search_dense_with_embedding(query_emb, top_k=fetch_k) if use_dense else []
    sparse_results = search_sparse(query_text, top_k=fetch_k) if use_sparse else []

    doc_map = {}
    rrf_scores = {}
    sparse_weight = 1.0 - dense_weight

    # Only use dense_weight when hybrid
    both_on = use_dense and use_sparse
    d_w = dense_weight if both_on else 1.0
    s_w = sparse_weight if both_on else 1.0

    for rank, hit in enumerate(dense_results, start=1):
        d_id = hit["doc_id"]
        doc_map[d_id] = (hit["document"], hit["metadata"])
        rrf_scores[d_id] = rrf_scores.get(d_id, 0.0) + (d_w / (k_rrf + rank))

    for rank, hit in enumerate(sparse_results, start=1):
        d_id = hit["doc_id"]
        if d_id not in doc_map:
            doc_map[d_id] = (hit["document"], hit["metadata"])
        rrf_scores[d_id] = rrf_scores.get(d_id, 0.0) + (s_w / (k_rrf + rank))

    candidate_doc_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:fetch_k]
    if not candidate_doc_ids:
        return []

    if not use_rerank:
        top_ids = candidate_doc_ids[:top_k]
        return [
            {
                "doc_id": d_id,
                "rerank_score": rrf_scores[d_id],
                "document": doc_map[d_id][0],
                "metadata": doc_map[d_id][1],
            }
            for d_id in top_ids
        ]

    candidate_pairs = [[query_text, doc_map[d_id][0]] for d_id in candidate_doc_ids]
    rerank_scores = reranker_model.predict(candidate_pairs)

    scored_candidates = [
        {
            "doc_id": d_id,
            "rerank_score": float(score),
            "document": doc_map[d_id][0],
            "metadata": doc_map[d_id][1],
        }
        for d_id, score in zip(candidate_doc_ids, rerank_scores)
    ]
    scored_candidates = sorted(scored_candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

    return scored_candidates


In [ ]:
import pandas as pd
from tqdm import tqdm
import gc, torch, csv, os


def run_custom_evaluation(
    df: pd.DataFrame,
    query_embeddings: list,
    top_k: int = 3,
    fetch_k: int = 25,
    use_dense: bool = True,
    use_sparse: bool = True,
    use_rerank: bool = True,
    output_path: str = "rag_custom_evaluation_report.csv",
    overwrite: bool = True,
):
    """
    Evaluation on the whole test set
    """
    if overwrite and os.path.exists(output_path):
        os.remove(output_path)

    citation_hits = 0
    format_scores = []
    rouge_l_scores = []

    queries = df["query_to_embed"].tolist()
    gt_answers = df["ground_truth_ans"].tolist()
    gt_law_names = df["law_name"].astype(str).str.strip().tolist()
    gt_law_ids = df["law_id"].astype(str).str.strip().tolist()

    fieldnames = [
        "id", "query", "ground_truth_citation",
        "citation_accurate", "format_adherence", "rouge_l", "generated_answer"
    ]

    write_header = not os.path.exists(output_path)
    csv_file = open(output_path, mode="a", newline="", encoding="utf-8")
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    if write_header:
        writer.writeheader()

    print(
        f"Bắt đầu chấm điểm {len(df)} mẫu | "
        f"dense={use_dense} sparse={use_sparse} rerank={use_rerank} "
        f"(top_k={top_k}, fetch_k={fetch_k})"
    )

    try:
        for idx in tqdm(range(len(df))):
            q_text = queries[idx]
            q_emb = query_embeddings[idx]
            gt_ans = gt_answers[idx]
            target_name = gt_law_names[idx]
            target_id = gt_law_ids[idx]

            # 1. Truy xuất — cấu hình tùy theo ablation (dense/sparse/rerank bật hay tắt)
            retrieved_docs = hybrid_search_ablation(
                query_text=q_text,
                query_emb=q_emb,
                top_k=top_k,
                fetch_k=fetch_k,
                use_dense=use_dense,
                use_sparse=use_sparse,
                use_rerank=use_rerank,
            )

            prompt = build_rag_prompt(q_text, retrieved_docs)
            pred_ans = generate_answer(prompt)

            is_citation_correct = check_citation_accuracy(pred_ans, target_name, target_id)
            if is_citation_correct:
                citation_hits += 1

            format_score = check_format_adherence(pred_ans)
            format_scores.append(format_score)

            r_score = scorer.score(gt_ans, pred_ans)
            rouge_l = r_score['rougeL'].fmeasure
            rouge_l_scores.append(rouge_l)

            writer.writerow({
                "id": df.iloc[idx].get("id", idx),
                "query": q_text,
                "ground_truth_citation": f"{target_id} - {target_name}",
                "citation_accurate": is_citation_correct,
                "format_adherence": format_score,
                "rouge_l": rouge_l,
                "generated_answer": pred_ans
            })

            if (idx + 1) % 10 == 0:
                csv_file.flush()

            if (idx + 1) % 20 == 0:
                gc.collect()
                torch.cuda.empty_cache()
    finally:
        csv_file.close()

    total_samples = len(df)
    avg_citation_acc = (citation_hits / total_samples) * 100
    avg_format_adh = (sum(format_scores) / total_samples) * 100
    avg_rouge_l = (sum(rouge_l_scores) / total_samples) * 100

    summary = {
        "citation_accuracy": avg_citation_acc,
        "format_adherence": avg_format_adh,
        "rouge_l": avg_rouge_l,
        "n_samples": total_samples,
    }

    print("\n" + "=" * 45)
    print("           BẢNG TỔNG KẾT ĐÁNH GIÁ           ")
    print("=" * 45)
    print(f"Citation Accuracy   : {avg_citation_acc:.2f}% ({citation_hits}/{total_samples})")
    print(f"Format Adherence    : {avg_format_adh:.2f}%")
    print(f"ROUGE-L Trung bình  : {avg_rouge_l:.2f}%")
    print("=" * 45)

    results_df = pd.read_csv(output_path)
    return results_df, summary


In [ ]:
def run_ablation_study(
    df: pd.DataFrame,
    query_embeddings: list,
    configs: list = None,
    top_k: int = 3,
    fetch_k: int = 25,
    output_dir: str = "ablation_results",
):

    os.makedirs(output_dir, exist_ok=True)

    if configs is None:
        configs = [
            {"name": "dense_only",         "use_dense": True,  "use_sparse": False, "use_rerank": False},
            {"name": "sparse_only",        "use_dense": False, "use_sparse": True,  "use_rerank": False},
            {"name": "hybrid_no_rerank",   "use_dense": True,  "use_sparse": True,  "use_rerank": False},
            {"name": "hybrid_with_rerank", "use_dense": True,  "use_sparse": True,  "use_rerank": True},
        ]

    all_summaries = []
    for cfg in configs:
        name = cfg["name"]
        print(f"\n>>> Đang chạy cấu hình: {name}")
        out_path = os.path.join(output_dir, f"{name}.csv")

        _, summary = run_custom_evaluation(
            df=df,
            query_embeddings=query_embeddings,
            top_k=top_k,
            fetch_k=fetch_k,
            use_dense=cfg["use_dense"],
            use_sparse=cfg["use_sparse"],
            use_rerank=cfg["use_rerank"],
            output_path=out_path,
            overwrite=True,
        )
        summary["config"] = name
        all_summaries.append(summary)

    summary_df = pd.DataFrame(all_summaries)[
        ["config", "citation_accuracy", "format_adherence", "rouge_l", "n_samples"]
    ]
    summary_df.to_csv(os.path.join(output_dir, "ablation_summary.csv"), index=False)

    print("\n" + "=" * 60)
    print("                 KẾT QUẢ ABLATION STUDY                ")
    print("=" * 60)
    print(summary_df.to_string(index=False))

    return summary_df


In [ ]:
# # Thực thi ablation study
# del model
# gc.collect()
# torch.cuda.empty_cache()

# ablation_summary_df = run_ablation_study(
#     df=test_df,
#     query_embeddings=query_embeddings,
#     top_k=3,
#     fetch_k=25,
# )
# ablation_summary_df


In [ ]:
import pandas as pd
from tqdm import tqdm


def evaluate_retrieval_ablation(
    df: pd.DataFrame,
    query_embeddings: list,
    configs: list = None,
    top_k_list: list = [1, 3, 5],
    fetch_k: int = 25,
):

    if configs is None:
        configs = [
            {"name": "dense_only",         "use_dense": True,  "use_sparse": False, "use_rerank": False},
            {"name": "sparse_only",        "use_dense": False, "use_sparse": True,  "use_rerank": False},
            {"name": "hybrid_no_rerank",   "use_dense": True,  "use_sparse": True,  "use_rerank": False},
            {"name": "hybrid_with_rerank", "use_dense": True,  "use_sparse": True,  "use_rerank": True},
        ]

    max_k = max(top_k_list)
    target_ids = df["target_doc_id"].tolist()
    queries = df["query_to_embed"].tolist()
    total = len(df)

    all_rows = []
    for cfg in configs:
        name = cfg["name"]
        hits = {k: 0 for k in top_k_list}
        mrr_total = 0.0

        print(f"\n>>> Đang đánh giá retrieval với cấu hình: {name}")
        for q_text, q_emb, target_id in tqdm(zip(queries, query_embeddings, target_ids), total=total):
            results = hybrid_search_ablation(
                query_text=q_text,
                query_emb=q_emb,
                top_k=max_k,
                fetch_k=fetch_k,
                use_dense=cfg["use_dense"],
                use_sparse=cfg["use_sparse"],
                use_rerank=cfg["use_rerank"],
            )
            retrieved_ids = [res["doc_id"] for res in results]

            for k in top_k_list:
                if target_id in retrieved_ids[:k]:
                    hits[k] += 1
            if target_id in retrieved_ids:
                rank = retrieved_ids.index(target_id) + 1
                mrr_total += 1.0 / rank

        row = {"config": name}
        for k in top_k_list:
            row[f"hit@{k}"] = hits[k] / total * 100
        row[f"mrr@{max_k}"] = mrr_total / total
        row["n_samples"] = total
        all_rows.append(row)

    summary_df = pd.DataFrame(all_rows)

    print("\n" + "=" * 60)
    print("           KẾT QUẢ ABLATION STUDY - RETRIEVAL ONLY        ")
    print("=" * 60)
    print(summary_df.to_latex())

    return summary_df


In [ ]:
# #Chạy full retrieval-only ablation trên toàn bộ test set (chạy sau khi đã kiểm tra ổn với 10 mẫu)
# retrieval_ablation_full = evaluate_retrieval_ablation(
#     df=test_df,
#     query_embeddings=query_embeddings,
#     top_k_list=[1, 3, 5],
#     fetch_k=25,
# )



In [ ]:
def run_zero_shot_evaluation(
    df: pd.DataFrame,
    output_path: str = "zero_shot_evaluation_report.csv",
    overwrite: bool = True,
):

    records = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        query = row["query_to_embed"]
        gt_answer = row["ground_truth_ans"]

        prompt = f"""Bạn là trợ lý pháp lý. Hãy trả lời câu hỏi sau, trích dẫn tên luật và điều luật liên quan nếu biết.

Câu hỏi: {query}

Trả lời:"""

        pred_answer = generate_answer(prompt)

        rouge_l = scorer.score(gt_answer, pred_answer)["rougeL"].fmeasure

        citation_ok = check_citation_accuracy(
            pred_answer,
            row.get("law_name", ""),
            row.get("law_id", ""),
        )

        records.append({
            "query": query,
            "ground_truth": gt_answer,
            "prediction": pred_answer,
            "rougeL": rouge_l,
            "citation_correct": citation_ok,
        })

    result_df = pd.DataFrame(records)
    if overwrite or not os.path.exists(output_path):
        result_df.to_csv(output_path, index=False)

    print(f"Mean ROUGE-L: {result_df['rougeL'].mean():.4f}")
    print(f"Citation accuracy: {result_df['citation_correct'].mean():.4f}")
    return result_df

In [ ]:
run_zero_shot_evaluation(test_df)